# Chronic Kidney Disease (CKD) Detection
**Data Science + Machine Learning mini project**

**Type:** supervised ML, binary classification (CKD = 1, Not CKD = 0)

**Steps:** Load → Inspect → Clean → EDA → Preprocess → Split → Train → Tune → Evaluate → Conclude

**Dataset:** `kidney_disease.csv` (same columns as the UCI / Kaggle CKD dataset). Put it in the same folder as this notebook.

### Column guide
`age` age · `bp` blood pressure · `sg` specific gravity · `al` albumin · `su` sugar · `rbc` red blood cells · `pc` pus cell · `pcc` pus cell clumps · `ba` bacteria · `bgr` blood glucose random · `bu` blood urea · `sc` serum creatinine · `sod` sodium · `pot` potassium · `hemo` hemoglobin · `pcv` packed cell volume · `wc` white cell count · `rc` red cell count · `htn` hypertension · `dm` diabetes mellitus · `cad` coronary artery disease · `appet` appetite · `pe` pedal edema · `ane` anemia · `classification` **target**

## 1. Import libraries

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             ConfusionMatrixDisplay, RocCurveDisplay, classification_report)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Load the data

In [ ]:
df = pd.read_csv("kidney_disease.csv")
print("Shape:", df.shape)
df.head()

## 3. Inspect the raw data

In [ ]:
df.info()
print("\nMissing values per column:")
print(df.isnull().sum().sort_values(ascending=False))
print("\nDuplicates:", df.duplicated().sum())
print("\nTarget values:", df["classification"].unique())
print("dm values    :", df["dm"].unique())

**Problems found**
- `pcv`, `wc`, `rc` should be numbers but are stored as text (they contain `?`)
- Hidden tabs/spaces in text columns (`"\tno"`, `" yes"`, `"ckd\t"`) make one category look like several
- Many missing values (`rbc`, `rc`, `wc`, `pot`, `sod` are the worst)
- An `id` column that is useless for prediction

## 4. Data cleaning

In [ ]:
clean = df.copy()

# 1) drop id column
clean = clean.drop(columns=["id"], errors="ignore")

# 2) strip tabs/spaces from every text column and turn '?' into missing
text_cols = clean.select_dtypes(include=["object", "string"]).columns
for c in text_cols:
    clean[c] = clean[c].astype(str).str.strip().replace({"?": np.nan, "nan": np.nan, "": np.nan})

# 3) convert columns that should be numeric
num_cols = ["age", "bp", "sg", "al", "su", "bgr", "bu", "sc", "sod", "pot", "hemo", "pcv", "wc", "rc"]
for c in num_cols:
    clean[c] = pd.to_numeric(clean[c], errors="coerce")

# 4) target: ckd -> 1, notckd -> 0
clean["classification"] = clean["classification"].map({"ckd": 1, "notckd": 0})

# 5) remove duplicates and rows with no target
clean = clean.drop_duplicates().dropna(subset=["classification"]).reset_index(drop=True)
clean["classification"] = clean["classification"].astype(int)

print("Shape after cleaning:", clean.shape)
print("\nTarget counts:\n", clean["classification"].value_counts())
print("\nMissing % per column:")
print((clean.isnull().mean() * 100).round(1).sort_values(ascending=False).head(10))
clean.to_csv("kidney_disease_clean.csv", index=False)

> We **do not fill missing values yet**. Filling them before the train/test split would leak test-set information. Imputation happens inside the pipeline (Section 7).

## 5. Exploratory Data Analysis (EDA)

In [ ]:
cat_cols = [c for c in clean.columns if c not in num_cols + ["classification"]]

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
clean["classification"].value_counts().sort_index().plot.bar(ax=ax[0], color=["#4C72B0", "#C44E52"])
ax[0].set_title("Class balance"); ax[0].set_xticklabels(["Not CKD", "CKD"], rotation=0)
(clean[num_cols].isnull().mean() * 100).sort_values().plot.barh(ax=ax[1], color="#55A868")
ax[1].set_title("Missing % (numeric columns)"); plt.tight_layout(); plt.show()

In [ ]:
clean[num_cols].hist(bins=20, figsize=(14, 9), color="#4C72B0"); plt.tight_layout(); plt.show()

In [ ]:
# Key medical features split by class
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), ["hemo", "sc", "bu", "pcv", "rc", "sg", "al", "bgr"]):
    sns.boxplot(data=clean, x="classification", y=col, ax=ax, palette=["#4C72B0", "#C44E52"])
    ax.set_xticklabels(["Not CKD", "CKD"])
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), ["htn", "dm", "appet", "pe", "ane", "rbc", "pc", "cad"]):
    sns.barplot(data=clean, x=col, y="classification", ax=ax, errorbar=None, palette="viridis")
    ax.set_ylabel("CKD rate")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(clean[num_cols + ["classification"]].corr(), annot=True, fmt=".1f", cmap="coolwarm", center=0)
plt.title("Correlation heatmap"); plt.show()

**EDA findings (write these in your own words):**
- Patients with CKD have **lower hemoglobin, PCV and red cell count**
- They have **higher serum creatinine, blood urea and albumin**, and lower specific gravity
- Hypertension, diabetes, poor appetite, pedal edema and anemia are far more common in CKD patients

## 6. Train / test split\nStratified 80/20, so both sets keep the same CKD ratio.

In [ ]:
X = clean.drop(columns="classification")
y = clean["classification"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("CKD rate  train: %.3f  test: %.3f" % (y_train.mean(), y_test.mean()))

## 7. Preprocessing pipeline\nMissing values filled and features scaled/encoded using **training data only**.

In [ ]:
num_pipe = Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
cat_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                     ("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocess = ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)])

## 8. Train and compare models (5-fold cross-validation)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
}
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, model in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    acc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy")
    f1 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1")
    rows.append({"Model": name, "CV Accuracy": acc.mean(), "CV Acc Std": acc.std(), "CV F1": f1.mean()})
pd.DataFrame(rows).sort_values("CV F1", ascending=False).round(3)

## 9. Hyperparameter tuning (Random Forest)

In [ ]:
rf_pipe = Pipeline([("prep", preprocess), ("model", RandomForestClassifier(random_state=RANDOM_STATE))])
param_grid = {"model__n_estimators": [100, 200],
              "model__max_depth": [3, 5, None],
              "model__min_samples_leaf": [1, 2, 4]}
grid = GridSearchCV(rf_pipe, param_grid, cv=cv, scoring="f1", n_jobs=-1)
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best CV F1: %.3f" % grid.best_score_)

## 10. Final evaluation on the unseen test set

In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Accuracy :", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall   :", round(recall_score(y_test, y_pred), 3))
print("F1-score :", round(f1_score(y_test, y_pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y_test, y_prob), 3))
print("\n", classification_report(y_test, y_pred, target_names=["Not CKD", "CKD"]))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["Not CKD", "CKD"], ax=ax[0], cmap="Blues")
ax[0].set_title("Confusion matrix")
RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax[1]); ax[1].plot([0, 1], [0, 1], "k--")
ax[1].set_title("ROC curve"); plt.tight_layout(); plt.show()

## 11. Feature importance

In [ ]:
names = best_model.named_steps["prep"].get_feature_names_out()
imp = pd.Series(best_model.named_steps["model"].feature_importances_, index=names)
imp.sort_values().tail(12).plot.barh(figsize=(8, 5), color="#4C72B0")
plt.title("Top 12 features"); plt.show()

## 12. Save the model and predict for a new patient

In [ ]:
joblib.dump(best_model, "ckd_model.joblib")

new_patient = pd.DataFrame([{
    "age": 60, "bp": 90, "sg": 1.010, "al": 3, "su": 1, "rbc": "abnormal", "pc": "abnormal",
    "pcc": "present", "ba": "notpresent", "bgr": 180, "bu": 85, "sc": 4.2, "sod": 130, "pot": 5.0,
    "hemo": 9.5, "pcv": 28, "wc": 9000, "rc": 3.4, "htn": "yes", "dm": "yes", "cad": "no",
    "appet": "poor", "pe": "yes", "ane": "yes"}])

loaded = joblib.load("ckd_model.joblib")
prob = loaded.predict_proba(new_patient)[0, 1]
print("Probability of CKD: %.1f%%  ->  %s" % (prob * 100, "CKD likely" if prob >= 0.5 else "Not CKD"))

## 13. Conclusion
- Cleaning fixed numbers stored as text, hidden tabs/spaces in categories, `?` placeholders, duplicates and missing values.
- EDA showed CKD is linked to low hemoglobin/PCV/RBC and high creatinine, urea and albumin, plus hypertension and diabetes.
- Preprocessing ran **inside a pipeline after the split**, so there is no data leakage.
- Three models were compared by cross-validation, the Random Forest was tuned, and the final model was tested once on unseen data.
- Recall matters most in medicine, because missing a sick patient is the costly mistake.

**Limitations:** small dataset (about 400 rows) and very separable classes, so accuracy is high and may not hold on bigger, messier real-world data. This is a learning project, not a diagnostic tool.

**Future work:** try XGBoost, add SHAP explanations, validate on a larger dataset, build a Streamlit app.